In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, confusion_matrix

model = joblib.load('models/lgbm_delay_classifier_final.pkl')

FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY',
    'DISTANCE', 'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_precip_1hr',
    'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_precip_1hr',
    'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year',
    'origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h',
    'origin_stress_index', 'real_time_turn_gap',
    'tail_delays_today', 'tail_active_hours',
    'origin_pressure_drop_stress', 'airport_fatigue_index',
]
CAT_FEATURES = ['OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label']

flights = pd.read_parquet('dataset/merged_flights_fe_v2.parquet')
for col in CAT_FEATURES:
    flights[col] = flights[col].astype('category')

test = flights[flights['FL_DATE'] >= '2025-01-01']
X_test = test[FEATURES]
y_test = test['ARR_DEL15']
y_pred = model.predict_proba(X_test)[:, 1]

y_bin = (y_pred >= 0.65).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_bin).ravel()

print(f"CONFUSION MATRIX (threshold 0.65):")
print(f"  True Positives (correctly flagged):    {tp:>10,}")
print(f"  False Positives (false alarms):        {fp:>10,}")
print(f"  True Negatives (correctly cleared):    {tn:>10,}")
print(f"  False Negatives (MISSED delays):       {fn:>10,}")

print(f"\nMISSED DELAYS ANALYSIS ({fn:,} flights):")

missed = test[(y_bin == 0) & (y_test == 1)]
caught = test[(y_bin == 1) & (y_test == 1)]

print(f"\n  Missed vs Caught delay characteristics:")
for col in ['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'LATE_AIRCRAFT_DELAY']:
    missed_pct = (missed[col].fillna(0) > 0).mean() * 100
    caught_pct = (caught[col].fillna(0) > 0).mean() * 100
    print(f"    {col:<25} Missed: {missed_pct:.1f}%   Caught: {caught_pct:.1f}%")

print(f"\n  Missed delays — avg prediction prob: {y_pred[(y_bin == 0) & (y_test == 1)].mean():.3f}")
print(f"  Caught delays — avg prediction prob: {y_pred[(y_bin == 1) & (y_test == 1)].mean():.3f}")

missed_arr = test.loc[missed.index, 'ARR_DELAY']
print(f"\n  Missed delay severity:")
print(f"    Mean: {missed_arr.mean():.1f} min")
print(f"    Median: {missed_arr.median():.1f} min")
print(f"    % just barely delayed (15-30 min): {((missed_arr >= 15) & (missed_arr <= 30)).mean()*100:.1f}%")
print(f"    % moderate (30-60 min): {((missed_arr > 30) & (missed_arr <= 60)).mean()*100:.1f}%")
print(f"    % severe (>60 min): {(missed_arr > 60).mean()*100:.1f}%")

delayed = test[test['ARR_DEL15'] == 1]
no_warning = delayed[delayed['cascade_score'] == 0]
print(f"\n{'=' * 60}")
print(f"58% UNPREDICTABLE PROOF:")
print(f"  Total delayed flights: {len(delayed):,}")
print(f"  With zero cascade warning: {len(no_warning):,} ({len(no_warning)/len(delayed)*100:.1f}%)")

carrier = no_warning['CARRIER_DELAY'].fillna(0).sum()
nas = no_warning['NAS_DELAY'].fillna(0).sum()
late = no_warning['LATE_AIRCRAFT_DELAY'].fillna(0).sum()
weather = no_warning['WEATHER_DELAY'].fillna(0).sum()
security = no_warning['SECURITY_DELAY'].fillna(0).sum()
total = carrier + nas + late + weather + security

print(f"\n  Surprise delay causes:")
print(f"    CARRIER_DELAY:        {carrier/total*100:.1f}%")
print(f"    NAS_DELAY:            {nas/total*100:.1f}%")
print(f"    LATE_AIRCRAFT_DELAY:  {late/total*100:.1f}%")
print(f"    WEATHER_DELAY:        {weather/total*100:.1f}%")
print(f"    SECURITY_DELAY:       {security/total*100:.1f}%")
print(f"\n  CARRIER + NAS = {(carrier+nas)/total*100:.1f}% of surprise delays")

print(f"\nAUC BY CARRIER:")
for carrier in sorted(test['OP_UNIQUE_CARRIER'].unique()):
    mask = test['OP_UNIQUE_CARRIER'] == carrier
    if y_test[mask].nunique() > 1:
        carrier_auc = roc_auc_score(y_test[mask], y_pred[mask])
        carrier_delay = y_test[mask].mean() * 100
        count = mask.sum()
        print(f"  {carrier:<5} AUC: {carrier_auc:.4f}  delay%: {carrier_delay:.1f}%  flights: {count:,}")

del flights, test

CONFUSION MATRIX (threshold 0.65):
  True Positives (correctly flagged):       638,975
  False Positives (false alarms):           250,079
  True Negatives (correctly cleared):     3,231,211
  False Negatives (MISSED delays):          398,861

MISSED DELAYS ANALYSIS (398,861 flights):

  Missed vs Caught delay characteristics:
    CARRIER_DELAY             Missed: 59.8%   Caught: 47.1%
    WEATHER_DELAY             Missed: 5.3%   Caught: 8.4%
    NAS_DELAY                 Missed: 60.4%   Caught: 46.3%
    LATE_AIRCRAFT_DELAY       Missed: 16.8%   Caught: 71.5%

  Missed delays — avg prediction prob: 0.391
  Caught delays — avg prediction prob: 0.914

  Missed delay severity:
    Mean: 56.7 min
    Median: 30.0 min
    % just barely delayed (15-30 min): 50.8%
    % moderate (30-60 min): 27.4%
    % severe (>60 min): 21.9%

58% UNPREDICTABLE PROOF:
  Total delayed flights: 1,037,836
  With zero cascade warning: 602,377 (58.0%)

  Surprise delay causes:
    CARRIER_DELAY:        42.9%
   